In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

In [2]:
DB_USER = "postgres"
DB_PASSWORD = "Postgres123!"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_dw"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

ModuleNotFoundError: No module named 'psycopg2'

In [ ]:
query = """
SELECT
    stock_code,
    year,
    month,
    orders,
    revenue,
    aov,
    customers,
    revenue_lag1,
    revenue_lag2,
    revenue_lag3,
    rolling_3m_avg,
    rolling_6m_avg
FROM ml.train_feature_monthly_sales
ORDER BY stock_code, year, month
"""

df = pd.read_sql(query, engine)
df.head()

In [ ]:
df = df.sort_values(["stock_code", "year", "month"]).copy()

df["target_revenue_next_month"] = (
    df.groupby("stock_code")["revenue"].shift(-1)
)

df = df.dropna(subset=["target_revenue_next_month"]).copy()
df.head()

In [ ]:
feature_cols = [
    "orders",
    "revenue",
    "aov",
    "customers",
    "revenue_lag1",
    "revenue_lag2",
    "revenue_lag3",
    "rolling_3m_avg",
    "rolling_6m_avg",
    "month"
]

X = df[feature_cols]
y = df["target_revenue_next_month"]

In [ ]:
df["yyyymm"] = df["year"] * 100 + df["month"]

cutoff = df["yyyymm"].quantile(0.8)

train_idx = df["yyyymm"] <= cutoff
test_idx = df["yyyymm"] > cutoff

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]

meta_test = df.loc[test_idx, ["stock_code", "year", "month"]].copy()

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, pred)
mape = np.mean(np.abs((y_test - pred) / np.maximum(y_test, 1))) * 100

print("MAE:", round(mae, 2))
print("MAPE:", round(mape, 2))

In [ ]:
result = meta_test.copy()
result["actual_revenue"] = y_test.values
result["predicted_revenue"] = pred
result["abs_error"] = np.abs(result["actual_revenue"] - result["predicted_revenue"])
result["ape"] = (
    result["abs_error"] / np.maximum(result["actual_revenue"], 1)
) * 100

# 간단한 pseudo interval
result["lower_ci"] = result["predicted_revenue"] * 0.9
result["upper_ci"] = result["predicted_revenue"] * 1.1
result["model_name"] = "random_forest"

result.head()

In [ ]:
result.to_csv("../result/prediction_monthly_sales.csv", index=False)

In [ ]:
result.to_sql(
    "prediction_monthly_sales",
    engine,
    schema="ml",
    if_exists="append",
    index=False
)

In [ ]:
chart_df = result.groupby(["year", "month"], as_index=False)[
    ["actual_revenue", "predicted_revenue"]
].sum()

chart_df["date"] = pd.to_datetime(
    chart_df["year"].astype(str) + "-" + chart_df["month"].astype(str).str.zfill(2) + "-01"
)

plt.figure(figsize=(10, 5))
plt.plot(chart_df["date"], chart_df["actual_revenue"], label="Actual")
plt.plot(chart_df["date"], chart_df["predicted_revenue"], label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Revenue")
plt.tight_layout()
plt.savefig("../result/forecast_actual_vs_pred.png")
plt.show()